In [1]:
!pip install librosa soundfile pandas numpy scikit-learn


In [2]:
import os

BASE_PATH = r"C:\Users\Simran\Downloads\archive (1)\LA\LA"

TRAIN_AUDIO_DIR = os.path.join(BASE_PATH, "ASVspoof2019_LA_train", "flac")
DEV_AUDIO_DIR = os.path.join(BASE_PATH, "ASVspoof2019_LA_dev", "flac")

TRAIN_PROTOCOL = os.path.join(BASE_PATH, "ASVspoof2019_LA_cm_protocols", "ASVspoof2019.LA.cm.train.trn.txt")
DEV_PROTOCOL = os.path.join(BASE_PATH, "ASVspoof2019_LA_cm_protocols", "ASVspoof2019.LA.cm.dev.trl.txt")

OUTPUT_DIR = "clean_dataset"
CLEAN_GENUINE_DIR = os.path.join(OUTPUT_DIR, "genuine")
CLEAN_SPOOF_DIR = os.path.join(OUTPUT_DIR, "spoof")

os.makedirs(CLEAN_GENUINE_DIR, exist_ok=True)
os.makedirs(CLEAN_SPOOF_DIR, exist_ok=True)

print("Folders ready.")
print("Train audio dir exists:", os.path.exists(TRAIN_AUDIO_DIR))
print("Dev audio dir exists:", os.path.exists(DEV_AUDIO_DIR))
print("Train protocol exists:", os.path.exists(TRAIN_PROTOCOL))
print("Dev protocol exists:", os.path.exists(DEV_PROTOCOL))

Folders ready.
Train audio dir exists: True
Dev audio dir exists: True
Train protocol exists: True
Dev protocol exists: True


In [3]:
import pandas as pd

def parse_protocol(protocol_path, audio_dir):
    columns = ["speaker_id", "filename", "system_id", "null_col", "label"]
    df = pd.read_csv(protocol_path, sep=" ", names=columns)
    df["filepath"] = df["filename"].apply(lambda x: os.path.join(audio_dir, x + ".flac"))
    df["label_binary"] = df["label"].apply(lambda x: 0 if x == "bonafide" else 1)
    return df[["filename", "filepath", "label", "label_binary", "system_id"]]

train_df = parse_protocol(TRAIN_PROTOCOL, TRAIN_AUDIO_DIR)
dev_df = parse_protocol(DEV_PROTOCOL, DEV_AUDIO_DIR)

print("Train samples:", len(train_df))
print("Dev samples:", len(dev_df))
train_df.head()

Train samples: 25380
Dev samples: 24844


,filename,filepath,label,label_binary,system_id
0,LA_T_1138215,C:\Users\Simran\Downloads\archive (1)\LA\LA\AS...,bonafide,0,-
1,LA_T_1271820,C:\Users\Simran\Downloads\archive (1)\LA\LA\AS...,bonafide,0,-
2,LA_T_1272637,C:\Users\Simran\Downloads\archive (1)\LA\LA\AS...,bonafide,0,-
3,LA_T_1276960,C:\Users\Simran\Downloads\archive (1)\LA\LA\AS...,bonafide,0,-
4,LA_T_1341447,C:\Users\Simran\Downloads\archive (1)\LA\LA\AS...,bonafide,0,-


In [4]:
N_PER_CLASS = 1000  # jitna chahiye utna badal sakti ho

def balanced_subset(df, n_per_class, seed=42):
    genuine = df[df["label"] == "bonafide"]
    spoof = df[df["label"] == "spoof"]

    n_genuine = min(n_per_class, len(genuine))
    n_spoof = min(n_per_class, len(spoof))

    genuine_sample = genuine.sample(n_genuine, random_state=seed)
    spoof_sample = spoof.sample(n_spoof, random_state=seed)

    return pd.concat([genuine_sample, spoof_sample]).sample(frac=1, random_state=seed).reset_index(drop=True)

train_subset = balanced_subset(train_df, N_PER_CLASS)
dev_subset = balanced_subset(dev_df, N_PER_CLASS // 4)

print("Train subset:", train_subset["label"].value_counts().to_dict())
print("Dev subset:", dev_subset["label"].value_counts().to_dict())

Train subset: {'spoof': 1000, 'bonafide': 1000}
Dev subset: {'spoof': 250, 'bonafide': 250}


In [5]:
import librosa

def check_valid_files(df, sample_check=True):
    valid_rows = []
    for idx, row in df.iterrows():
        if not os.path.exists(row["filepath"]):
            continue
        if sample_check:
            try:
                y, sr = librosa.load(row["filepath"], sr=None, duration=0.1)
                if len(y) == 0:
                    continue
            except Exception:
                continue
        valid_rows.append(idx)
    return df.loc[valid_rows].reset_index(drop=True)

train_subset = check_valid_files(train_subset)
dev_subset = check_valid_files(dev_subset)

print("Valid train samples:", len(train_subset))
print("Valid dev samples:", len(dev_subset))

Valid train samples: 2000
Valid dev samples: 500


In [6]:
import numpy as np
import soundfile as sf

TARGET_SR = 16000
TARGET_DURATION = 4  # seconds
TARGET_LENGTH = TARGET_SR * TARGET_DURATION

def clean_and_save_audio(row):
    try:
        y, sr = librosa.load(row["filepath"], sr=TARGET_SR)

        # remove silence
        y, _ = librosa.effects.trim(y, top_db=20)

        # normalize volume
        max_val = np.max(np.abs(y))
        if max_val > 0:
            y = y / max_val

        # fix duration
        y = librosa.util.fix_length(y, size=TARGET_LENGTH)

        # decide output folder
        out_dir = CLEAN_GENUINE_DIR if row["label"] == "bonafide" else CLEAN_SPOOF_DIR
        out_path = os.path.join(out_dir, row["filename"] + ".wav")

        sf.write(out_path, y, TARGET_SR)
        return out_path
    except Exception as e:
        print(f"Failed: {row['filename']} -> {e}")
        return None

In [7]:
train_subset["clean_path"] = train_subset.apply(clean_and_save_audio, axis=1)
dev_subset["clean_path"] = dev_subset.apply(clean_and_save_audio, axis=1)

train_subset = train_subset.dropna(subset=["clean_path"]).reset_index(drop=True)
dev_subset = dev_subset.dropna(subset=["clean_path"]).reset_index(drop=True)

print("Cleaned train files:", len(train_subset))
print("Cleaned dev files:", len(dev_subset))

Cleaned train files: 2000
Cleaned dev files: 500


In [8]:
   %pip install resemblyzer --no-deps

Note: you may need to restart the kernel to use updated packages.


In [9]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [10]:
# =========================================================
#SPEAKER VERIFICATION / IMPERSONATION DETECTION
# =========================================================


import os
import numpy as np
from resemblyzer import VoiceEncoder, preprocess_wav

# ---------------------------------------------------------
# STEP 1: Setup — point to your existing clean_dataset folder
# ---------------------------------------------------------
OUTPUT_DIR = "clean_dataset"
GENUINE_DIR = os.path.join(OUTPUT_DIR, "genuine")
SPOOF_DIR = os.path.join(OUTPUT_DIR, "spoof")

encoder = VoiceEncoder()

# ---------------------------------------------------------
# STEP 2: Build a "reference identity" automatically
# We use a few genuine files as the registered voice
# (in the real demo this would be the CEO's own recorded voice,
#  but for testing on the dataset we simulate it this way)
# ---------------------------------------------------------
genuine_files = sorted(os.listdir(GENUINE_DIR))
reference_files = genuine_files[:3]          # first 3 genuine files = "registered speaker"
remaining_genuine = genuine_files[3:]         # rest used for testing "real match" cases

print("Reference (registered identity) files:", reference_files)

ref_embeds = []
for f in reference_files:
    wav = preprocess_wav(os.path.join(GENUINE_DIR, f))
    ref_embeds.append(encoder.embed_utterance(wav))

reference_embedding = np.mean(ref_embeds, axis=0)
np.save("reference_embedding.npy", reference_embedding)
print("Reference embedding created and saved.\n")

# ---------------------------------------------------------
# STEP 3: The core verification function
# This is the function your dashboard/team will call.
# ---------------------------------------------------------
def verify_speaker(audio_path, claimed_identity="CEO", threshold=0.75, reference_embedding=reference_embedding):
    """
    Compares an incoming audio file against the saved reference embedding.
    Returns the exact JSON format agreed in the team's integration contract.
    """
    test_wav = preprocess_wav(audio_path)
    test_embedding = encoder.embed_utterance(test_wav)

    similarity = np.dot(reference_embedding, test_embedding) / (
        np.linalg.norm(reference_embedding) * np.linalg.norm(test_embedding)
    )
    similarity = float(similarity)

    return {
        "speaker_match_score": round(similarity, 2),
        "speaker_verified": similarity >= threshold,
        "claimed_identity": claimed_identity
    }

# ---------------------------------------------------------
# STEP 4: Quick self-test — proves the module works
# Tests a few genuine files (should MATCH) and a few spoof
# files (should show IMPERSONATION / low match)
# ---------------------------------------------------------
print("=" * 55)
print("SELF-TEST: Genuine voices (expected -> MATCH)")
print("=" * 55)
for f in remaining_genuine[:5]:
    result = verify_speaker(os.path.join(GENUINE_DIR, f), claimed_identity="CEO")
    status = "MATCH" if result["speaker_verified"] else "IMPERSONATION"
    print(f"{f} -> Score: {result['speaker_match_score']*100:.1f}%  -> {status}")

print()
print("=" * 55)
print("SELF-TEST: Spoof/cloned voices (expected -> IMPERSONATION)")
print("=" * 55)
spoof_files = sorted(os.listdir(SPOOF_DIR))[:5]
for f in spoof_files:
    result = verify_speaker(os.path.join(SPOOF_DIR, f), claimed_identity="CEO")
    status = "MATCH" if result["speaker_verified"] else "IMPERSONATION"
    print(f"{f} -> Score: {result['speaker_match_score']*100:.1f}%  -> {status}")

print("\nModule ready. Use verify_speaker(audio_path, claimed_identity) for any new audio file.")

Loaded the voice encoder model on cpu in 0.09 seconds.
Reference (registered identity) files: ['LA_D_1060968.wav', 'LA_D_1086391.wav', 'LA_D_1130470.wav']
Reference embedding created and saved.

SELF-TEST: Genuine voices (expected -> MATCH)
LA_D_1156906.wav -> Score: 83.0%  -> MATCH
LA_D_1158837.wav -> Score: 64.0%  -> IMPERSONATION
LA_D_1199158.wav -> Score: 72.0%  -> IMPERSONATION
LA_D_1232613.wav -> Score: 62.0%  -> IMPERSONATION
LA_D_1245754.wav -> Score: 55.0%  -> IMPERSONATION

SELF-TEST: Spoof/cloned voices (expected -> IMPERSONATION)
LA_D_1039057.wav -> Score: 71.0%  -> IMPERSONATION
LA_D_1062092.wav -> Score: 68.0%  -> IMPERSONATION
LA_D_1071228.wav -> Score: 65.0%  -> IMPERSONATION
LA_D_1105502.wav -> Score: 58.0%  -> IMPERSONATION
LA_D_1127112.wav -> Score: 50.0%  -> IMPERSONATION

Module ready. Use verify_speaker(audio_path, claimed_identity) for any new audio file.
